# 06. Threshold Analysis & Operating-Point Evaluation: Heart Disease Screening

**Phase**: Heart Disease — Phase 6  
**Document Reference**: `docs/HEART_THRESHOLD_ANALYSIS.md`  
**Parent Artifacts**: `artifacts/heart/experiments/oof_predictions.csv`, `artifacts/heart/experiments/test_predictions.csv`  
**Dataset**: Cleveland Heart Disease (`datasets/raw/heart/processed.cleveland.data`)  
**Strict Phase Boundary**: Operating-point evaluation only. No probability calibration (Phase 7), no SHAP explainability (Phase 8), no model retraining, no production deployment.

---

### Section 1: Objective & Methodology
In medical diagnostic machine learning and clinical screening, the default threshold of $t = 0.50$ is an arbitrary computational convention. The clinical penalties of false negatives (missed coronary artery disease, resulting in unmanaged ischemia or infarction) and false positives (secondary testing anxiety or invasive angiography referral) are highly asymmetric.

**Core Principles of Phase 6**:
1. **Fit-on-Train / OOF Threshold Selection**: Candidate operating points are explored and selected **exclusively** on leak-free Out-Of-Fold (OOF) training predictions ($N=242$).
2. **Hold-Out Integrity**: The 61-sample test set is evaluated **strictly once and descriptively** to inspect the selected thresholds without selection leakage.
3. **Multi-Metric Scanning**: Evaluate 99 decision cutoffs $t \in [0.01, 0.99]$ across Sensitivity, Specificity, Precision, NPV, F1, Accuracy, and Youden's J.
4. **Target Screening Criterion**: Identify high-sensitivity candidate operating regions targeting $\ge 90\%$ sensitivity to prioritize disease capture.


### Section 2: Phase 5 Artifact Verification & Environment
Verify existence and checksum integrity of Phase 5 prediction artifacts and the underlying Cleveland raw dataset.


In [ ]:
import os
import sys
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid', font_scale=1.05)

# Add ml-service root to path
ml_service_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "ml-service"))
if ml_service_dir not in sys.path:
    sys.path.insert(0, ml_service_dir)

from app.heart_preprocessing import verify_raw_dataset_integrity
from app.heart_threshold_analysis import (
    MODEL_COLUMNS,
    DEFAULT_THRESHOLDS,
    validate_prediction_artifacts,
    calculate_threshold_metrics,
    analyze_threshold_grid,
    find_high_sensitivity_regions,
    find_balanced_operating_point,
    verify_default_threshold_results,
)

# 1. Raw Dataset Integrity Check
valid, md5_val, sha256_val, file_size = verify_raw_dataset_integrity()
print(f"Raw dataset integrity verified: {valid}")
print(f"File size: {file_size} bytes | MD5: {md5_val}")

# 2. Artifact Paths
oof_path = os.path.join(ml_service_dir, "artifacts", "heart", "experiments", "oof_predictions.csv")
test_path = os.path.join(ml_service_dir, "artifacts", "heart", "experiments", "test_predictions.csv")

print(f"OOF predictions exists:  {os.path.exists(oof_path)}")
print(f"Test predictions exists: {os.path.exists(test_path)}")


### Section 3: OOF Prediction Loading
Load the leak-free Out-Of-Fold predictions from Phase 5 (242 training observations across 5 repeats of 5-fold CV).


In [ ]:
df_oof, df_test = validate_prediction_artifacts(oof_path, test_path)

print(f"OOF predictions loaded: {len(df_oof)} samples")
print(f"Negative (0): {(df_oof['true_label'] == 0).sum()} | Positive (1): {(df_oof['true_label'] == 1).sum()}")
df_oof.head()


### Section 4: Hold-Out Test Prediction Loading
Load the locked hold-out test predictions (61 observations, strictly isolated during threshold exploration).


In [ ]:
print(f"Test predictions loaded: {len(df_test)} samples")
print(f"Negative (0): {(df_test['true_label'] == 0).sum()} | Positive (1): {(df_test['true_label'] == 1).sum()}")
df_test.head()


### Section 5: Probability Output Validation
Verify that all model probabilities are strictly continuous and bounded within $[0, 1]$ with no missing values.


In [ ]:
for key, info in MODEL_COLUMNS.items():
    oof_col = info['oof_col']
    test_col = info['test_prob_col']
    oof_p = df_oof[oof_col]
    test_p = df_test[test_col]
    
    print(f"[{info['name']}]")
    print(f"  OOF range:  [{oof_p.min():.4f}, {oof_p.max():.4f}] | Unique: {len(oof_p.unique())}")
    print(f"  Test range: [{test_p.min():.4f}, {test_p.max():.4f}] | Unique: {len(test_p.unique())}")
    assert (oof_p >= 0.0).all() and (oof_p <= 1.0).all()
    assert (test_p >= 0.0).all() and (test_p <= 1.0).all()


### Section 6: Threshold Evaluation Grid
Construct a fine-grained 99-step decision threshold grid $t \in [0.01, 0.99]$ with step size $\Delta t = 0.01$.
Classification Rule:
$$\hat{y} = \begin{cases} 1 & \text{if } P(Y=1 \mid X) \ge t \\ 0 & \text{if } P(Y=1 \mid X) < t \end{cases}$$


In [ ]:
print(f"Threshold grid: {len(DEFAULT_THRESHOLDS)} steps from {DEFAULT_THRESHOLDS[0]:.2f} to {DEFAULT_THRESHOLDS[-1]:.2f}")
print(f"Step size: {DEFAULT_THRESHOLDS[1] - DEFAULT_THRESHOLDS[0]:.2f}")


### Section 7: Metric Calculation Protocol
Calculate comprehensive diagnostic and operational metrics:
- **Sensitivity / Recall**: $TP / (TP + FN)$
- **Specificity**: $TN / (TN + FP)$
- **Precision / PPV**: $TP / (TP + FP)$
- **Negative Predictive Value (NPV)**: $TN / (TN + FN)$
- **Accuracy**: $(TP + TN) / N$
- **F1-Score**: $2 \times \frac{\text{Precision} \times \text{Sensitivity}}{\text{Precision} + \text{Sensitivity}}$
- **Youden's J Statistic**: $\text{Sensitivity} + \text{Specificity} - 1$


In [ ]:
y_oof = df_oof["true_label"].values
y_test = df_test["true_label"].values

oof_grids = {}
test_grids = {}

for key, info in MODEL_COLUMNS.items():
    name = info["name"]
    oof_grids[name] = analyze_threshold_grid(y_oof, df_oof[info["oof_col"]].values, model_name=name, cohort_type="OOF")
    test_grids[name] = analyze_threshold_grid(y_test, df_test[info["test_prob_col"]].values, model_name=name, cohort_type="Hold-Out Test")

print("Grid analysis completed for all 3 models across OOF and Hold-Out cohorts.")


### Section 8: Logistic Regression Threshold Analysis
Evaluate trade-offs across the threshold spectrum for Logistic Regression (Robust L1).


In [ ]:
lr_name = MODEL_COLUMNS["LogisticRegression"]["name"]
lr_oof = oof_grids[lr_name]
print(f"=== {lr_name}: Key Threshold Points (OOF N=242) ===")
print(lr_oof[lr_oof["threshold"].isin([0.10, 0.20, 0.31, 0.40, 0.50, 0.56, 0.70, 0.80])][
    ["threshold", "sensitivity", "specificity", "precision", "npv", "f1", "youden_j", "fn", "fp"]
].to_string(index=False))


### Section 9: Random Forest Threshold Analysis
Evaluate trade-offs across the threshold spectrum for Random Forest ($n=200, \text{leaf}=5$).


In [ ]:
rf_name = MODEL_COLUMNS["RandomForest"]["name"]
rf_oof = oof_grids[rf_name]
print(f"=== {rf_name}: Key Threshold Points (OOF N=242) ===")
print(rf_oof[rf_oof["threshold"].isin([0.10, 0.20, 0.32, 0.40, 0.50, 0.55, 0.70, 0.80])][
    ["threshold", "sensitivity", "specificity", "precision", "npv", "f1", "youden_j", "fn", "fp"]
].to_string(index=False))


### Section 10: XGBoost Threshold Analysis
Evaluate trade-offs across the threshold spectrum for XGBoost ($n=100, d=2, \eta=0.05$).


In [ ]:
xgb_name = MODEL_COLUMNS["XGBoost"]["name"]
xgb_oof = oof_grids[xgb_name]
print(f"=== {xgb_name}: Key Threshold Points (OOF N=242) ===")
print(xgb_oof[xgb_oof["threshold"].isin([0.10, 0.20, 0.24, 0.40, 0.50, 0.62, 0.70, 0.80])][
    ["threshold", "sensitivity", "specificity", "precision", "npv", "f1", "youden_j", "fn", "fp"]
].to_string(index=False))


### Section 11: High-Sensitivity Operating Regions (Sensitivity $\ge 0.90$)
Identify the upper boundary of the high-sensitivity screening zone on OOF data.


In [ ]:
for name in [lr_name, rf_name, xgb_name]:
    sub = find_high_sensitivity_regions(oof_grids[name], min_sensitivity=0.90)
    upper_cand = sub.iloc[-1]
    print(f"[{name}]")
    print(f"  High-Sensitivity Range: [{sub['threshold'].min():.2f}, {sub['threshold'].max():.2f}]")
    print(f"  Upper Candidate Cutoff: t = {upper_cand['threshold']:.2f}")
    print(f"  OOF Metrics: Sens={upper_cand['sensitivity']:.4f}, Spec={upper_cand['specificity']:.4f}, Prec={upper_cand['precision']:.4f}, FN={int(upper_cand['fn'])}, FP={int(upper_cand['fp'])}\n")


### Section 12: Balanced Operating Regions (Maximizing Youden's J)
Identify threshold cutoffs that maximize overall discriminatory balance ($J = \text{Sens} + \text{Spec} - 1$).


In [ ]:
for name in [lr_name, rf_name, xgb_name]:
    bal = find_balanced_operating_point(oof_grids[name])
    print(f"[{name}] Balanced Cutoff: t = {bal['threshold']:.2f}")
    print(f"  OOF Youden J:    {bal['youden_j']:.4f}")
    print(f"  OOF Sensitivity: {bal['sensitivity']:.4f}")
    print(f"  OOF Specificity: {bal['specificity']:.4f}")
    print(f"  OOF Accuracy:    {bal['accuracy']:.4f}\n")


### Section 13: Default 0.50 Baseline Verification
Verify exact reproduction of Phase 5 confusion matrix counts on the hold-out test set ($N=61$).


In [ ]:
def_check = verify_default_threshold_results(df_test)
for name, status in def_check.items():
    print(f"{name}: 100% Phase 5 Test Set Reproduction -> {status}")


### Section 14: Summary of Selected OOF Candidate Thresholds
Candidate operating points derived strictly from OOF training evidence:
- **Logistic Regression**: High-Sensitivity $t = 0.31$, Balanced $t = 0.56$, Default $t = 0.50$
- **Random Forest**: High-Sensitivity $t = 0.32$, Balanced $t = 0.55$, Default $t = 0.50$
- **XGBoost**: High-Sensitivity $t = 0.24$, Balanced $t = 0.62$, Default $t = 0.50$


In [ ]:
threshold_summary_path = os.path.join(ml_service_dir, "artifacts", "heart", "threshold", "threshold_summary.csv")
df_summary = pd.read_csv(threshold_summary_path)
df_summary[["model_name", "operating_point", "threshold", "oof_sensitivity", "oof_specificity", "oof_precision", "oof_fn", "oof_fp"]]


### Section 15: Descriptive Evaluation on Locked Hold-Out Test Set (N=61)
Evaluate candidate operating points strictly once on the untouched hold-out test set (33 negative, 28 positive).


In [ ]:
df_summary[["model_name", "operating_point", "threshold", "test_sensitivity", "test_specificity", "test_precision", "test_npv", "test_f1", "test_fn", "test_fp"]]


### Section 16: Threshold Stability & Curve Analysis
Inspect stability of metrics across neighboring cutoffs ($\Delta t = \pm 0.02$) to verify absence of erratic point discontinuities.


In [ ]:
# Display generated trade-off figures
from PIL import Image

thresh_dir = os.path.join(ml_service_dir, "artifacts", "heart", "threshold")
plot_files = [
    "sensitivity_specificity_vs_threshold.png",
    "precision_npv_vs_threshold.png",
    "false_positive_negative_vs_threshold.png",
    "youden_j_vs_threshold.png",
]

for p_file in plot_files:
    p_path = os.path.join(thresh_dir, p_file)
    if os.path.exists(p_path):
        img = Image.open(p_path)
        plt.figure(figsize=(15, 4.5))
        plt.imshow(img)
        plt.axis("off")
        plt.title(p_file, fontsize=12)
        plt.show()


### Section 17: Limitations & Clinical Guardrails
1. **Granularity on Small Datasets**: With 242 training observations and 111 positive cases, each patient represents $\sim 0.9\%$ sensitivity. Small metric jumps are natural sampling artifacts.
2. **Threshold Sensitivity vs. Specificity Trade-Off**: Achieving $\ge 96\%$ sensitivity on the test set increases false alarms from 4–5 to 9–12. In screening, secondary non-invasive follow-up (e.g. stress echocardiography) can resolve false positives.
3. **No Clinical Optimization**: These operating points represent empirical candidates, not medically validated clinical decision thresholds.
4. **Uncalibrated Model Scores**: Probabilities remain ranking scores. Calibration must precede true clinical risk interpretation.


### Section 18: Handoff to Phase 7 (Probability Calibration)
Phase 6 threshold analysis is complete.

**Deliverables Ready for Phase 7**:
- `artifacts/heart/threshold/threshold_metrics_oof.csv`: Full 99-step threshold curves on training data.
- `artifacts/heart/threshold/threshold_metrics_test.csv`: 99-step hold-out test evaluation curves.
- `artifacts/heart/threshold/threshold_summary.csv`: Summary of operating points across OOF and test cohorts.
- `artifacts/heart/threshold/default_threshold_comparison.csv`: Baseline comparison table.

**Next Step (Phase 7)**:
- Investigate whether model-estimated probabilities require isotonic regression or Platt scaling (sigmoid calibration) to align predicted probabilities with empirical event frequencies.
